# Lindblad simulation for the ATS circuit

The goal of this notebook is to investigate the dynamics of the memory mode and see if chaos appears, as it appears in classical mechanics

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
import Plot_setting
from Plot_setting import *
from Utils import * 
import dynamiqs as dq
import jax.numpy as jnp
from scipy.special import jv
import matplotlib.pyplot as plt

In [ ]:
ħ=1
e=1
phi_quantum=1/2

#### Caracterization of the circuit #####

w_a = 3.7*2*jnp.pi
w_b_0 = 7.05*2*jnp.pi

Lamb_shift = - 0.05 #Lamb shift in GHz

w_b = w_b_0 #+ Lamb_shift*2*jnp.pi # Shifted frequency of mode b

phi_a = 0.11
Z_a = 2*phi_quantum**2*phi_a**2
C_a_inv = Z_a*w_a
E_Ca = C_a_inv/2

phi_b = 0.2
Z_b = 2*phi_quantum**2*phi_b**2
C_b_inv = Z_b*w_b
E_Cb = C_b_inv/2

E_J = 37*2*jnp.pi




kappa=1/10; # 1/ns from Leon's paper

w_d=w_b # Drive frequency
w_p=2*w_a-w_d # Pump Frequency

epsilon_0 = 0.6*jnp.pi
g2 = jv(1,epsilon_0)*E_J*phi_a**2*phi_b # (Leon's paper)
epsilon_d=6 *g2#0.75*g2_max

kappa_2 = 4*g2**2/kappa

In [ ]:
Z_b

## Simulation of the decay of one oscillator with no drive

In [ ]:
n_a = 4
psi_0 = dq.fock(n_a, n_a-1)

T=2/kappa
# initialize operators, initial state and saving times
a = dq.destroy(n_a)
H = w_a * a.dag() @ a
jump_ops = [jnp.sqrt(kappa) * a]

dt = 1 / (2 * w_a) # Time step for the simulation
tsave = jnp.arange(0,T, dt)
# run simulation
result = dq.mesolve(H, jump_ops, psi_0, tsave)
print(result)

In [ ]:
n_mean = dq.expect(dq.number(n_a), result.states).real


In [ ]:
result.states[1]

In [ ]:
n_th = (n_a-1)*jnp.exp(-kappa*tsave)

In [ ]:
plt.plot(tsave, n_mean, label='Mean number of excitations')
plt.plot(tsave, n_th, ls='--', label='Theoretical number of excitations')
plt.xlabel('Time')
plt.ylabel('Number of excitations')
plt.legend()

## Simulation of a driven oscillator

In [ ]:
epsilon_0 = 0.6#*jnp.pi
g2 = jv(1,epsilon_0)*E_J*phi_a**2*phi_b # (Leon's paper)
epsilon_d=2 *g2#0.75*g2_max


n_b = 10
psi_0 = dq.fock(n_b, 0)

T=10/kappa
# initialize operators, initial state and saving times
b = dq.destroy(n_b)

w_d=w_b_0
drive = lambda t: epsilon_d * jnp.cos(w_d * t)

H_drive = dq.modulated(drive, b+b.dag())

H = w_b_0 * b.dag() @ b + H_drive

jump_ops = [jnp.sqrt(2*kappa) * b]

dt = 1 / (5 * w_b_0) # Time step for the simulation
tsave = jnp.arange(0,T, dt)
# run simulation
result = dq.mesolve(H, jump_ops, psi_0, tsave)
print(result)

In [ ]:
n_mean = dq.expect(dq.number(n_b), result.states).real
b_mean = dq.expect(b, result.states)

b_mean_re = jnp.real(b_mean)
b_mean_im = jnp.imag(b_mean)

In [ ]:
b_mean_th = epsilon_d/(2j*kappa)*(1-jnp.exp(-kappa*tsave))*jnp.exp(-1j*w_d*tsave)
b_mean_th_re = jnp.real(b_mean_th)
b_mean_th_im = jnp.imag(b_mean_th)

n_mean_th = epsilon_d**2/(4*kappa**2)*(1-jnp.exp(-kappa*tsave))**2

In [ ]:
fig = plt.figure(figsize=(figsize[0]*3, figsize[1]))
ax_1, ax_2, ax_3 = fig.subplots(1, 3)

ax_1.plot(tsave*kappa, n_mean, label='n')
ax_1.plot(tsave*kappa, n_mean_th, ls='--', label='theoretical')
ax_1.legend()
ax_2.plot(tsave*kappa, b_mean_re, label='Re(b)')
ax_2.plot(tsave*kappa, b_mean_th_re, label='theoretical', alpha=0.5)
ax_2.legend()
ax_3.plot(tsave*kappa, b_mean_im, label='Im(b)')
ax_3.plot(tsave*kappa, b_mean_th_im, label='theoretical', alpha=0.5)
ax_3.legend()


In [ ]:
FFT_freq, FFT_re_b  = fft_perso(b_mean_re, tsave)

max_FFT = FFT_freq[jnp.argmax(FFT_re_b)]
fig = plt.figure()
ax = fig.add_subplot(111)

ax.plot(FFT_freq, FFT_re_b, label='FFT of Re(b)')
ax.axvline(max_FFT, color='r', linestyle='--', label=str(max_FFT))
ax.set_yscale('log')
ax.set_xlim(6.8, 7.2)
#ax.set_ylim(1e-4,1)
ax.legend()
plt.show()

## Simulation of the ATS up to third order

In [ ]:
kappa = 0.1 *2*np.pi #

In [ ]:
g2/2/jnp.pi*1000

In [ ]:
epsilon_0 = 0.12#*jnp.pi
g2 = jv(1,epsilon_0)*E_J*phi_a**2*phi_b # (Leon's paper)
epsilon_d=10 *g2#0.75*g2_max
kappa_2 = 4*g2**2/kappa

T=135/kappa
n_a = 14
n_b=(n_a+1)//2
# initialize operators, initial state and saving times
a = dq.destroy(n_a)
b = dq.destroy(n_b)


varphi_a_op = jnp.sqrt(Z_a/2)*(a + a.dag()) / phi_quantum
varphi_b_op = jnp.sqrt(Z_b/2)*(b + b.dag()) / phi_quantum

varphi_a_op_tot = dq.tensor(varphi_a_op, dq.eye(n_b))
varphi_b_op_tot = dq.tensor(dq.eye(n_a), varphi_b_op)


alpha = jnp.sqrt(epsilon_d/(2*g2))
cat_state = (dq.coherent(n_a, alpha) + dq.coherent(n_a, -alpha))/jnp.sqrt(2)
psi_0 = dq.tensor(dq.fock(n_a, 1), dq.fock(n_b, 0))

w_bis = 6 * jnp.pi * kappa / 5
w_d = 7.05*2*jnp.pi # Drive frequency
drive = lambda t: epsilon_d * jnp.cos(w_d * t) 
H_drive = dq.modulated(drive, dq.tensor(dq.eye(n_a),b+b.dag()))


w_p = 2*w_a - w_d
pump = lambda t: - 2 * E_J *jnp.sin(epsilon_0 * jnp.cos(w_p * t))
H_pump = dq.modulated(pump, dq.sinm(varphi_a_op_tot + varphi_b_op_tot) - varphi_a_op_tot - varphi_b_op_tot)


H_a = dq.tensor(w_a * dq.number(n_a), dq.eye(n_b))
H_b = dq.tensor(dq.eye(n_a), w_b_0 * dq.number(n_b))

H = H_a + H_b + H_drive + H_pump

jump_ops = [jnp.sqrt(2*kappa) * dq.tensor(dq.eye(n_a), b)]

dt = 1 / (10 * max(w_a, w_b, abs(w_d))) # Time step for the simulation
tsave = jnp.arange(0,T, dt)

In [ ]:
epsilon_0 = 0.5*jnp.pi
g2 = jv(1,epsilon_0)*E_J*phi_a**2*phi_b # (Leon's paper)
epsilon_d=10 *g2#0.75*g2_max
kappa_2 = 4*g2**2/kappa

alpha = jnp.sqrt(epsilon_d/(2*g2))

T=5/kappa
n_a = 20
n_b=(n_a+1)//2
# initialize operators, initial state and saving times
a = dq.destroy(n_a)
b = dq.destroy(n_b)


varphi_a_op = jnp.sqrt(Z_a/2)*(a + a.dag()) / phi_quantum
varphi_b_op = jnp.sqrt(Z_b/2)*(b + b.dag()) / phi_quantum

varphi_a_op_tot = dq.tensor(varphi_a_op, dq.eye(n_b))
varphi_b_op_tot = dq.tensor(dq.eye(n_a), varphi_b_op)



cat_state = (dq.coherent(n_a, alpha) + dq.coherent(n_a, -alpha))/jnp.sqrt(2)
psi_0 = dq.tensor(dq.fock(n_a, 1), dq.fock(n_b, 0))

w_bis = 6 * jnp.pi * kappa / 5
w_d = 7.05*2*jnp.pi # Drive frequency
drive = lambda t: epsilon_d * jnp.cos(w_d * t) 
H_drive = dq.modulated(drive, dq.tensor(dq.eye(n_a),b+b.dag()))


w_p = 2*w_a - w_d
pump = lambda t: - 2 * E_J *jnp.sin(epsilon_0 * jnp.cos(w_p * t))
H_pump = dq.modulated(pump, dq.sinm(varphi_a_op_tot + varphi_b_op_tot) - varphi_a_op_tot - varphi_b_op_tot)


H_a = dq.tensor(w_a * dq.number(n_a), dq.eye(n_b))
H_b = dq.tensor(dq.eye(n_a), w_b_0 * dq.number(n_b))

H = H_a + H_b + H_drive + H_pump

jump_ops = [jnp.sqrt(2*kappa) * dq.tensor(dq.eye(n_a), b)]

dt = 1 / (5 * max(w_a, w_b, abs(w_d))) # Time step for the simulation
tsave = jnp.linspace(0,T, 10000)

In [ ]:
# run simulation
result = dq.mesolve(H, jump_ops, psi_0, tsave, method=dq.method.Tsit5(max_steps=20000000))
print(result)

In [ ]:
state_a = [dq.ptrace(state, 0) for state in result.states]

In [ ]:
w_eff = (w_d + w_p) / 2
U_list = [dq.expm(1j * w_eff * dq.number(n_a) * t) for t in tsave]
state_a_rotating = [U @ s @ U.dag() for U, s in zip(U_list, state_a)]


In [ ]:
theta = np.linspace(0,2*np.pi, 100)

fig ,ax = plt.subplots(figsize=figsize_car)
dq.plot.wigner(state_a_rotating[-1], ax = ax)
ax.plot(alpha * np.cos(theta), alpha * np.sin(theta), color='r', 
        linewidth = 1, linestyle = '--')

In [ ]:
overlap = []
for th in theta:
    angle_shift = np.exp(1j*th)
    expected_cat = (dq.coherent(n_a, angle_shift*alpha) - dq.coherent(n_a, -angle_shift*alpha)) / jnp.sqrt(2)
    overlap.append(dq.overlap(expected_cat, state_a_rotating[-1]))
plt.plot(theta, overlap)

In [ ]:
theta[np.argmax(overlap)]

In [ ]:
max(overlap)

In [ ]:
np.pi/2

In [ ]:
angle_shift = np.exp(1j*1.6501294746128208)
expected_cat = (dq.coherent(n_a, angle_shift*alpha) - dq.coherent(n_a, -angle_shift*alpha)) / jnp.sqrt(2)

dq.plot.wigner(expected_cat)


In [ ]:
dq.plot.wigner_gif(state_a_rotating, fps=25, gif_duration = 10.0)

In [ ]:
aadag_mean = dq.expect(dq.tensor(a@a.dag(), dq.eye(n_b)), result.states)
adaga_mean = dq.expect(dq.tensor(a.dag()@a, dq.eye(n_b)), result.states)

bbdag_mean = dq.expect(dq.tensor(dq.eye(n_a), b@b.dag()), result.states)
bdagb_mean = dq.expect(dq.tensor(dq.eye(n_a), b.dag()@b), result.states)


In [ ]:

plt.plot(tsave*kappa_2,1-(aadag_mean-adaga_mean).real, label = r'1-$\langle [a, a^\dagger] \rangle$')
plt.plot(tsave*kappa_2,1-(bbdag_mean-bdagb_mean).real, label = r'1-$\langle [b, b^\dagger] \rangle$', alpha=0.9)
plt.legend()
plt.xlabel(r"$\kappa_2 t$")
plt.show()

In [ ]:
a_mean_re = jnp.real(a_mean)
a_mean_im = jnp.imag(a_mean)

b_mean_re = jnp.real(b_mean)
b_mean_im = jnp.imag(b_mean)

In [ ]:
b_mean_th = epsilon_d/(2*jnp.sqrt(2))/(1j*kappa)*(1-jnp.exp(-kappa*tsave))*jnp.exp(-1j*w_d*tsave)
b_mean_th_re = jnp.real(b_mean_th)
b_mean_th_im = jnp.imag(b_mean_th)

n_mean_th = epsilon_d**2/(8*kappa**2)*(1-jnp.exp(-kappa*tsave))**2

In [ ]:
b_mean

In [ ]:
a_mean_rotating = a_mean * jnp.exp(-1j * (w_a+w_bis) * tsave)
a_mean_re_rotating = jnp.real(a_mean_rotating)
a_mean_im_rotating = jnp.imag(a_mean_rotating)

In [ ]:
plt.plot(a_mean_re_rotating, a_mean_im_rotating, label='Mean value of a ', lw=0.5)

In [ ]:
fig = plt.figure(figsize=(figsize_car[0]*2, figsize_car[1]))
ax_1, ax_2 = fig.subplots(1, 2)

ax_1.plot(a_mean_re, a_mean_im, label='Mean value of a', lw=0.5)
ax_2.plot(a_mean_re_rotating, a_mean_im_rotating, label='Mean value of a ', lw=0.5)

In [ ]:
fig = plt.figure(figsize=(figsize_car[0]*2, figsize_car[1]))
ax_1, ax_2 = fig.subplots(1, 2)

ax_1.plot(a_mean_re, a_mean_im, label='Mean value of a', lw=0.5)
ax_2.plot(a_mean_re_rotating, a_mean_im_rotating, label='Mean value of b', lw=0.5)


In [ ]:
FFT_freq, FFT_re_b = fft_perso(b_mean_re, tsave)
max_FFT = FFT_freq[jnp.argmax(FFT_re_b)]
fig = plt.figure()
ax = fig.add_subplot(111)

ax.plot(FFT_freq, FFT_re_b, label='FFT of Re(b)')
ax.axvline(max_FFT, color='r', linestyle='--', label=str(max_FFT))
ax.axvline(7.1, color='g', linestyle='--', label=str(max_FFT))
ax.set_yscale('log')
ax.set_xlim(6.8,8.2)
#ax.set_ylim(1e-4,1)
ax.set_title('FFT of Re(b)')
ax.set_xlabel('Frequency (GHz)')
ax.set_ylabel('Amplitude')
plt.show()

In [ ]:
max_FFT

In [ ]:
FFT_freq, FFT_re_a  = fft_perso(a_mean_re, tsave)
max_FFT = FFT_freq[jnp.argmax(FFT_re_a)]
fig = plt.figure()
ax = fig.add_subplot(111)

ax.plot(FFT_freq, FFT_re_a, label='FFT of Re(a)')
ax.axvline(3.7, color='r', linestyle='--', label=str(max_FFT))
ax.set_yscale('log')
ax.set_xlim(0,15)
#ax.set_ylim(1e-4,1)
ax.set_title('FFT of Re(a)')
ax.set_xlabel('Frequency (GHz)')
ax.set_ylabel('Amplitude')
plt.show()

In [ ]:
max_FFT = FFT_freq[jnp.argmax(FFT_re_a)]
fig = plt.figure()
ax = fig.add_subplot(111)

ax.plot(FFT_freq, FFT_re_a, label='FFT of Re(a)')
#ax.axvline(max_FFT, color='r', linestyle='--', label=str(max_FFT))
ax.set_yscale('log')
ax.set_xlim(0, 15)
#ax.set_ylim(1e-4,1)
ax.set_title('FFT of Re(a)')
ax.set_xlabel('Frequency (GHz)')
ax.set_ylabel('Amplitude')
plt.show()

In [ ]:
import json
b_mean_dict = [{"real": x.real, "imag": x.imag} for x in b_mean.tolist()]
with open('../../Output/b_mean_dict.json', 'w') as f:
    json.dump(b_mean_dict, f)

In [ ]:
plt.plot(n_b_mean)

In [ ]:
plt.plot(b_mean_re)

# We try to look directly in the rotating frame

In [ ]:
epsilon_0 = 0.2#*jnp.pi
g2 = jv(1,epsilon_0)*E_J*phi_a**2*phi_b # (Leon's paper)
epsilon_d=6 *g2#0.75*g2_max

T=10/kappa
n_a = 20
n_b=(n_a+1)//2
# initialize operators, initial state and saving times
a = dq.destroy(n_a)
b = dq.destroy(n_b)

varphi_a_op = jnp.sqrt(Z_a/2)*(a + a.dag()) / phi_quantum
varphi_b_op = jnp.sqrt(Z_b/2)*(b + b.dag()) / phi_quantum

varphi_a_op_tot = dq.tensor(varphi_a_op, dq.eye(n_b))
varphi_b_op_tot = dq.tensor(dq.eye(n_a), varphi_b_op)

psi_0 = dq.tensor(dq.fock(n_a, 0), dq.fock(n_b, 0))

w_d = 7.05*2*jnp.pi # Drive frequency
drive = lambda t: epsilon_d * jnp.cos(w_d * t) 
H_drive = dq.modulated(drive, dq.tensor(dq.eye(n_a),b+b.dag()))

w_p = 2*w_a - w_d
pump = lambda t: - 2 * E_J *jnp.sin(epsilon_0 * jnp.cos(w_p * t))
H_pump = dq.modulated(pump, dq.sinm(varphi_a_op_tot + varphi_b_op_tot) - varphi_a_op_tot - varphi_b_op_tot)


H_a = dq.tensor(w_a * a.dag() @ a, dq.eye(n_b))
H_b = dq.tensor(dq.eye(n_a), w_b_0 * b.dag() @ b)

H = H_b + H_drive + H_pump

jump_ops = [jnp.sqrt(2*kappa) * dq.tensor(dq.eye(n_a), b)]

dt = 1 / (10 * max(w_a, w_b, abs(w_d))) # Time step for the simulation
tsave = jnp.arange(0,T, dt)

In [ ]:
# run simulation
result = dq.mesolve(H, jump_ops, psi_0, tsave)
print(result)

In [ ]:
state_a = [dq.ptrace(state, 0) for state in result.states]

In [ ]:
dq.plot.wigner_gif(state_a, fps=25)